# InsureAssist — LoRA Fine-Tuning (Phase 2)

**Goal:** teach a small open LLM to answer insurance questions in our style, cheaply.

**LoRA (Low-Rank Adaptation):** freeze the big model, train only a tiny "adapter" (a few MB).
Runs on a **free Colab T4 GPU** in minutes. We load the model in fp16 (fits the T4's 15 GB).

**Why this notebook is robust:** it uses Colab's own pre-installed, mutually-consistent
`transformers` / `datasets` / `numpy`, and only adds `peft` + `mlflow`. Training uses the
stable Hugging Face `Trainer` (no version-fragile extras).

### How to run
1. Open in Colab:
   `https://colab.research.google.com/github/mzquadri/insureassist-rag-mlops/blob/main/finetune/lora_finetune.ipynb`
2. **Runtime → Change runtime type → T4 GPU → Save.**
3. **Runtime → Run all.** When done, download the `adapter/` folder (Phase 3 uses it).

You'll learn: Hugging Face `transformers`, `peft` (LoRA), the `Trainer`, and MLflow.

## 1. Install only what's missing
We deliberately do **not** pin/downgrade transformers, datasets, or numpy — that avoids the
binary-incompatibility errors that come from fighting Colab's environment.

In [ ]:
# One-time environment setup.
# Installs peft + mlflow-skinny and ensures a consistent NumPy 2.x. If this runtime's
# NumPy was previously downgraded (binary-incompatibility error), this cell fixes it and
# restarts the kernel ONCE. If it restarts, just click Runtime -> Run all again.
!pip -q install -U "numpy>=2,<3" peft mlflow-skinny 2>/dev/null

try:
    import numpy, datasets  # noqa: F401
    print("Environment OK - numpy", numpy.__version__)
except Exception:
    print("Dependencies fixed; restarting runtime. When it reconnects, click Run all again.")
    import IPython
    IPython.Application.instance().kernel.do_shutdown(True)


## 2. Check the GPU (must show a Tesla T4)

In [ ]:
import torch
print('GPU available:', torch.cuda.is_available())
!nvidia-smi -L

## 3. Load the base model (fp16) + tokenizer
**Phi-3-mini** is a small, open model. fp16 + gradient checkpointing keeps training within
the T4's memory.

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

BASE_MODEL = "microsoft/Phi-3-mini-4k-instruct"

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    torch_dtype=torch.float16,
    device_map="auto",
    trust_remote_code=True,
    attn_implementation="eager",   # avoids flash-attn warnings on T4
)
model.config.use_cache = False       # required with gradient checkpointing
model.enable_input_require_grads()   # required for LoRA + gradient checkpointing
print("Base model loaded (fp16).")

## 4. Training data → formatted text
Each (question, answer) pair becomes a chat, rendered to one `text` string with the model's
chat template. (A real project uses thousands of examples; this small set shows the workflow.)

In [ ]:
from datasets import Dataset

raw = [
    {"q": "Does home insurance cover water damage from a burst pipe?",
     "a": "Yes. Sudden and accidental water damage from a burst pipe is covered, including tracing and accessing the leak. Gradual leakage or wear and tear is not covered."},
    {"q": "What is the excess for an escape of water claim?",
     "a": "A higher excess of EUR 500 applies to escape-of-water claims, versus the standard EUR 250."},
    {"q": "Up to how much is jewellery covered per item?",
     "a": "Jewellery is covered up to a single-item limit of EUR 2,000 unless separately listed on the schedule."},
    {"q": "How long do I have to report a home claim?",
     "a": "Home insurance claims must be reported within 30 days of the incident."},
    {"q": "Does comprehensive auto cover include a courtesy car?",
     "a": "Yes, up to 14 days while your vehicle is repaired by an approved garage."},
    {"q": "What extra excess applies to drivers under 25?",
     "a": "An additional young-driver excess of EUR 300 on top of the EUR 400 own-damage excess."},
    {"q": "What is the maximum no-claims discount?",
     "a": "Up to 65%, reached after five claim-free years."},
    {"q": "Is mechanical breakdown covered by auto insurance?",
     "a": "No, mechanical or electrical breakdown is excluded."},
    {"q": "Is damage covered if the home is empty for two months?",
     "a": "No. Damage is excluded if the home is unoccupied for more than 60 consecutive days."},
    {"q": "What is needed before a car theft claim is processed?",
     "a": "A police report reference number is required."},
]

def to_text(ex):
    messages = [
        {"role": "system", "content": "You are a precise insurance policy assistant."},
        {"role": "user", "content": ex["q"]},
        {"role": "assistant", "content": ex["a"]},
    ]
    return {"text": tokenizer.apply_chat_template(messages, tokenize=False)}

ds = Dataset.from_list([to_text(r) for r in raw])
print(ds)
print(ds[0]["text"][:400])

## 5. Tokenize + attach the LoRA adapter
We tokenize the text, then wrap the model with a LoRA adapter. `target_modules` are Phi-3's
attention projection layers; only those tiny adapter weights are trained.

In [ ]:
from peft import LoraConfig, get_peft_model

def tokenize(ex):
    return tokenizer(ex["text"], truncation=True, max_length=512)

tokenized = ds.map(tokenize, remove_columns=ds.column_names)

lora = LoraConfig(
    r=16, lora_alpha=32, lora_dropout=0.05, bias="none",
    task_type="CAUSAL_LM",
    target_modules=["qkv_proj", "o_proj"],   # Phi-3 attention projections
)
model = get_peft_model(model, lora)
model.print_trainable_parameters()   # shows how few params LoRA trains

## 6. Train with the Hugging Face Trainer + log to MLflow
The `Trainer` is the stable, standard training loop. `DataCollatorForLanguageModeling`
handles padding and builds the labels. MLflow records params, loss, and the adapter.

In [ ]:
import mlflow
from transformers import Trainer, TrainingArguments, DataCollatorForLanguageModeling

collator = DataCollatorForLanguageModeling(tokenizer, mlm=False)

args = TrainingArguments(
    output_dir="adapter",
    num_train_epochs=10,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    logging_steps=1,
    save_strategy="no",
    fp16=True,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    report_to=[],
)

mlflow.set_experiment("insureassist-lora")
with mlflow.start_run():
    mlflow.log_params({
        "base_model": BASE_MODEL, "r": lora.r, "lora_alpha": lora.lora_alpha,
        "epochs": args.num_train_epochs, "lr": args.learning_rate,
    })
    trainer = Trainer(
        model=model, args=args, train_dataset=tokenized, data_collator=collator,
    )
    trainer.train()
    final_loss = trainer.state.log_history[-1].get("train_loss")
    if final_loss is not None:
        mlflow.log_metric("final_train_loss", final_loss)
    model.save_pretrained("adapter")             # saves ONLY the small LoRA adapter
    mlflow.log_artifacts("adapter", artifact_path="lora_adapter")
    print("Training done. Adapter saved to ./adapter")

## 7. Quick test — does it answer in our style?

In [ ]:
model.config.use_cache = True   # re-enable for faster generation
from transformers import pipeline

pipe = pipeline("text-generation", model=model, tokenizer=tokenizer)
messages = [
    {"role": "system", "content": "You are a precise insurance policy assistant."},
    {"role": "user", "content": "Is a burst pipe covered by home insurance?"},
]
out = pipe(messages, max_new_tokens=120, do_sample=False)
print(out[0]["generated_text"][-1]["content"])

## 8. Download the adapter
In the Colab file browser (folder icon, left), right-click the **`adapter`** folder →
**Download**. Put it in your repo at `finetune/adapter/` for Phase 3.

**Next:** Phase 3 — plug this adapter into the RAG pipeline and evaluate with RAGAS.